# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object
meta = dataset.metadata

print(f"Dataset Name: {meta.name}\nDescription: {meta.description}")
print(f"License: {meta.license}\nVersion: {meta.version}")
if hasattr(meta, 'keywords'):
    print(f"Keywords: {meta.keywords}")
print(f"Date Published: {getattr(meta, 'datePublished', 'N/A')}")

## 2. Data Overview

Next, we will review record sets, their IDs, and fields available in the dataset. For each record set, we'll list its `@id`, `name`, and its field `@id`s and names. This allows you to reference entities using their `@id`.

In [ ]:
# List all record set IDs and their fields
from pprint import pprint

print("Record Sets available in this dataset:\n")
rs_objs = list(dataset.record_sets)
record_set_ids = []

for rs in rs_objs:
    print(f"- recordSet @id: {rs.id}")
    print(f"  name: {rs.name}")
    record_set_ids.append(rs.id)
    fields = list(rs.fields)
    if fields:
        print(f"  Fields in this recordSet:")
        for f in fields:
            print(f"    - field @id: {f.id} | name: {getattr(f, 'name', None)} | dataType: {getattr(f, 'dataType', None)}")
    else:
        print("  No fields found.")
    print()

# Show collected recordSet @ids
print("All discovered recordSet @ids:")
print(record_set_ids)

## 3. Data Extraction

Load data from one or more record sets into DataFrames for analysis. Entities are referenced by their `@id`. We'll extract all record sets discovered above.

In [ ]:
# Extract data from all available record sets into DataFrames
dataframes = {}
for rsid in record_set_ids:
    # Croissant datasets may have empty/non-tabular record sets, skip gracefully
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for {rsid}")
        print(f"Columns: {df.columns.tolist()}")
        print()
    except Exception as e:
        print(f"Could not load record set {rsid}: {e}")

# Pick first record set with data for further analysis
main_record_set = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set = rsid
        break

if main_record_set:
    print(f"Main record set for EDA: {main_record_set}")
    print(dataframes[main_record_set].head())
else:
    print("No record set with data was found.")

## 4. Exploratory Data Analysis (EDA)

Now let's analyze one numeric field in a data-rich record set. We'll filter values, normalize a numeric column by its mean and standard deviation, and (if possible) group by a categorical field, referencing all fields by their `@id`.

In [ ]:
# Attempt to select a numeric field for analysis
df = dataframes[main_record_set]

# Find a numeric field (float/int columns only)
numeric_fields = []
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_fields.append(col)
        # Check for uniqueness (some colums are ids/strings that may be ints)

if not numeric_fields:
    print("No numeric fields found for EDA.")
else:
    print("Numeric fields detected:", numeric_fields)
    numeric_field_id = numeric_fields[0]
    # Choose threshold as 10th percentile value for demonstration
    threshold = df[numeric_field_id].quantile(0.10)
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try to group by a categorical (non-numeric, non-object) field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object and df[col].nunique() > 1 and df[col].nunique() < len(df)//2:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical group field detected for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field or compare groups if a grouping field is detected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_fields:
    print("No numeric fields found for visualization.")
else:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group field detected
    if 'group_field_id' in locals() and group_field_id:
        if group_field_id in df.columns:
            # Show boxplot by group
            plt.figure(figsize=(12,5))
            top_groups = df[group_field_id].value_counts().index[:10]
            subset = df[df[group_field_id].isin(top_groups)]
            sns.boxplot(data=subset, x=group_field_id, y=numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion

We loaded and explored the dataset using `mlcroissant`. We identified record sets and fields by their `@id`, extracted tabular data, filtered and normalized a numeric field, and visualized distributions. For further analysis and domain-specific operations, refer to the dataset's [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and documentation.